# 004 · Booleans and Truthiness

> **Syntax › Data Types** · Python 3.14

Python has a `bool` type with two values, `True` and `False` — but unlike C#, **every** object can be used where a boolean is expected. Each value is either *truthy* or *falsy*, and idiomatic Python leans on this heavily.

**In this notebook you will learn:**

- `True`/`False` and the fact that `bool` is a subclass of `int`
- which values are falsy, and the idiomatic `if items:` check
- how `bool()` decides, and how your own classes hook in via `__bool__` / `__len__`
- `and`/`or` returning operands (not just `bool`)
- `any()` and `all()` as the counterparts of LINQ's `Any()` and `All()`

## C# mental model

| C# | Python |
|---|---|
| `true` / `false` | `True` / `False` (capitalised) |
| `bool` is its own struct, no conversion to `int` | `bool` is a subclass of `int` — `True == 1` |
| `if (x)` requires `x` to be `bool` | `if x:` works for **any** object |
| `if (list.Count > 0)` | `if items:` |
| `!`, `&&`, `\|\|` | `not`, `and`, `or` |
| `list.Any()` / `list.All(p)` | `any(...)` / `all(...)` |

## `True` and `False`

The literals are capitalised — `true` is just an undefined name in Python. The logical operators are words: `and`, `or`, `not`. *(Operators › 002 · Comparison and Logical Operators covers these in depth.)*

In [ ]:
is_active = True
is_admin = False

print(is_active and not is_admin)   # C#: isActive && !isAdmin
print(type(True))                   # <class 'bool'>

try:
    true
except NameError as e:
    print(f"{type(e).__name__}: {e}")

## `bool` is a subclass of `int`

For historical reasons `True` and `False` are literally the integers `1` and `0` in disguise. In C# this would be a compile error (`true + true` doesn't compile, and `Convert.ToInt32(true)` is explicit).

In [ ]:
print(isinstance(True, int))   # True
print(True == 1, False == 0)   # True True
print(True + True)             # 2
print(True * 10)               # 10

This is occasionally useful — summing booleans counts how many are true:

In [ ]:
scores = [72, 45, 90, 38, 66]
passed = sum(score >= 50 for score in scores)    # each comparison is 0 or 1
print(passed)                                     # 3  (C#: scores.Count(s => s >= 50))

⚠️ **Gotcha:** because `bool` is an `int`, `isinstance(flag, int)` is `True` for booleans, and `True` and `1` are the *same* dict key:

In [ ]:
print(isinstance(False, int))      # True — surprising when validating input

d = {1: "one", True: "true"}
print(d)                           # {1: 'true'} — True == 1 and hash(True) == hash(1)

def is_real_int(x) -> bool:
    return isinstance(x, int) and not isinstance(x, bool)

print(is_real_int(5), is_real_int(True))   # True False

## Truthiness

In C#, `if` requires a `bool`. In Python, `if` (and `while`, `and`, `or`, `not`) call `bool()` on any value. The rule of thumb: **empty or zero is falsy, everything else is truthy**.

Falsy values:

| Category | Falsy values |
|---|---|
| Constants | `False`, `None` |
| Numbers | `0`, `0.0`, `0j`, `Decimal(0)`, `Fraction(0)` |
| Empty sequences / collections | `""`, `[]`, `()`, `{}`, `set()`, `range(0)`, `b""` |

In [ ]:
from decimal import Decimal

values = [False, None, 0, 0.0, 0j, Decimal(0), "", [], (), {}, set(), range(0)]
for v in values:
    print(f"{v!r:>12} -> {bool(v)}")

In [ ]:
truthy = [True, 1, -1, 0.001, "0", "False", " ", [0], [[]], (None,), {"": None}]
for v in truthy:
    print(f"{v!r:>14} -> {bool(v)}")

⚠️ **Gotcha:** the strings `"0"` and `"False"` are **truthy** — they are non-empty. `bool("False")` is `True`. To parse a string as a boolean you need explicit logic (C#'s `bool.Parse` has no built-in Python equivalent).

In [ ]:
def parse_bool(s: str) -> bool:
    match s.strip().lower():
        case "true" | "yes" | "1" | "on":
            return True
        case "false" | "no" | "0" | "off":
            return False
        case _:
            raise ValueError(f"not a boolean: {s!r}")

print(bool("False"), parse_bool("False"))   # True False

## The `if items:` idiom

PEP 8 recommends relying on truthiness for emptiness checks instead of comparing lengths.

```csharp
// C#
if (items.Count > 0) { ... }
if (!string.IsNullOrEmpty(name)) { ... }
```

In [ ]:
items = []
name = "alice"

if not items:                 # rather than: if len(items) == 0:
    print("no items")

if name:                      # rather than: if name != "":
    print(f"hello {name}")

⚠️ **Gotcha:** truthiness cannot distinguish "missing" from "empty/zero". If `0` or `""` are valid values, test explicitly with `is None` *(see Data Types › 005 · None)*:

In [ ]:
def describe(count: int | None) -> str:
    if not count:                   # BUG: treats 0 the same as None
        return "unknown"
    return f"{count} items"

def describe_fixed(count: int | None) -> str:
    if count is None:
        return "unknown"
    return f"{count} items"

print(describe(0), "|", describe_fixed(0))   # unknown | 0 items

## `and` / `or` return operands, not `bool`

`x or y` returns `x` if it is truthy, otherwise `y`. `x and y` returns `x` if it is falsy, otherwise `y`. Both short-circuit like `&&` / `||`. This makes `or` usable for defaults — with the same caveat as above.

In [ ]:
print("alice" or "anonymous")    # alice
print("" or "anonymous")         # anonymous
print(0 or 10)                   # 10   — careful if 0 is a valid value!
print([1] and "non-empty")       # non-empty
print(None and "never")          # None — short-circuits

## `bool()` and the `__bool__` / `__len__` hooks

`bool(x)` is the explicit conversion. For your own classes, Python asks:

1. `__bool__()` if defined — must return `True`/`False`
2. otherwise `__len__()` — zero means falsy
3. otherwise the object is truthy

C# has no direct equivalent (you could overload `operator true`/`operator false`, but it is rare). *(More in Classes & OOP › 007 · Dunder Methods.)*

In [ ]:
class Cart:
    def __init__(self, *products: str):
        self.products = list(products)

    def __len__(self) -> int:
        return len(self.products)

class Account:
    def __init__(self, balance: float):
        self.balance = balance

    def __bool__(self) -> bool:
        return self.balance > 0

class Plain:
    pass

print(bool(Cart()), bool(Cart("pen")))              # False True — via __len__
print(bool(Account(0)), bool(Account(10)))          # False True — via __bool__
print(bool(Plain()))                                # True — default

## `any()` and `all()`

`any(iterable)` returns `True` if at least one element is truthy; `all(iterable)` returns `True` if every element is truthy. They short-circuit, and they pair naturally with generator expressions — the equivalent of LINQ `Any(predicate)` / `All(predicate)`.

```csharp
// C#
bool hasNegative = numbers.Any(n => n < 0);
bool allPositive = numbers.All(n => n > 0);
```

In [ ]:
numbers = [3, 8, -2, 5]

print(any(n < 0 for n in numbers))    # True  (C#: numbers.Any(n => n < 0))
print(all(n > 0 for n in numbers))    # False (C#: numbers.All(n => n > 0))
print(any([0, "", None]))             # False — all falsy
print(all(["a", 1, [0]]))             # True  — all truthy

⚠️ **Gotcha:** on an empty iterable, `any()` is `False` and `all()` is `True` (vacuous truth) — exactly the same as LINQ.

In [ ]:
print(any([]), all([]))    # False True

## Summary

| Task | C# | Python |
|---|---|---|
| Literals | `true` / `false` | `True` / `False` |
| Logical ops | `&&` `\|\|` `!` | `and` `or` `not` |
| Non-empty check | `list.Count > 0` | `if items:` |
| Null-or-empty string | `string.IsNullOrEmpty(s)` | `if not s:` |
| Explicit conversion | `Convert.ToBoolean(x)` | `bool(x)` |
| Count matches | `xs.Count(p)` | `sum(p(x) for x in xs)` |
| Any match | `xs.Any(p)` | `any(p(x) for x in xs)` |
| All match | `xs.All(p)` | `all(p(x) for x in xs)` |
| Custom truthiness | `operator true` (rare) | `__bool__` / `__len__` |

## Exercises

1. Without running it, predict which of these are truthy: `0.0`, `"0"`, `[None]`, `{}`, `" "`, `-1`, `()`. Then check with `bool()`.
2. Given `orders = [{"id": 1, "paid": True}, {"id": 2, "paid": False}, {"id": 3, "paid": True}]`, use `any`, `all`, and `sum` to answer: is any order unpaid? are all paid? how many are paid?
3. Write a class `Inventory` that holds a dict of product → quantity and is falsy when the **total** quantity is zero (even if it has keys).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — truthy: "0", [None], " ", -1   falsy: 0.0, {}, ()
for v in [0.0, "0", [None], {}, " ", -1, ()]:
    print(repr(v), bool(v))

# 2
orders = [{"id": 1, "paid": True}, {"id": 2, "paid": False}, {"id": 3, "paid": True}]
print(any(not o["paid"] for o in orders))   # True
print(all(o["paid"] for o in orders))       # False
print(sum(o["paid"] for o in orders))       # 2 — True counts as 1

# 3
class Inventory:
    def __init__(self, stock: dict[str, int]):
        self.stock = stock

    def __bool__(self) -> bool:
        return sum(self.stock.values()) > 0

print(bool(Inventory({"pen": 0, "cup": 0})))   # False
print(bool(Inventory({"pen": 2})))             # True
```

</details>